In [ ]:
!pip install timm mlflow -q

# CONFIG — change model here to experiment
CONFIG = {
    'model':    'vit',   # 'vit' | 'efficientnet_b0' | 'efficientnet_b2'
    'epochs':   30,
    'mixup':    True,
    'cutmix':   True,
    'unfreeze': 4,       # ViT blocks to unfreeze
}

MODEL_SETTINGS = {
    'vit':             {'lr': 5e-6,  'batch': 16, 'img': 224, 'mean': [0.5,0.5,0.5],       'std': [0.5,0.5,0.5]},
    'efficientnet_b0': {'lr': 1e-4,  'batch': 32, 'img': 224, 'mean': [0.485,0.456,0.406], 'std': [0.229,0.224,0.225]},
    'efficientnet_b2': {'lr': 3e-4,  'batch': 32, 'img': 260, 'mean': [0.485,0.456,0.406], 'std': [0.229,0.224,0.225]},
}

import os, json, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.optim as optim, timm, warnings
warnings.filterwarnings('ignore')
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from google.colab import files

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = 4
S           = MODEL_SETTINGS[CONFIG['model']]

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Model: {CONFIG["model"]}')
print('Setup complete!')

In [ ]:
#2 - Download Dataset
print('Upload kaggle.json...')
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json')
os.system('pip install kaggle -q')
os.system('kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset')
os.system('unzip -q brain-tumor-mri-dataset.zip -d /content/data')
TRAIN_DIR = TEST_DIR = None
for root, dirs, _ in os.walk('/content/data'):
    for d in dirs:
        if d == 'Training': TRAIN_DIR = os.path.join(root, d)
        if d == 'Testing':  TEST_DIR  = os.path.join(root, d)
print(f'Train: {TRAIN_DIR}')
print(f'Test:  {TEST_DIR}')

In [ ]:
#3 - Train Model

def mixup(x, y, a=0.4):
    l = np.random.beta(a, a)
    idx = torch.randperm(x.size(0)).to(x.device)
    return l*x + (1-l)*x[idx], y, y[idx], l

def cutmix(x, y, a=1.0):
    l = np.random.beta(a, a)
    idx = torch.randperm(x.size(0)).to(x.device)
    _, _, H, W = x.shape
    cw = int(W * np.sqrt(1-l)); ch = int(H * np.sqrt(1-l))
    cx = np.random.randint(W); cy = np.random.randint(H)
    x1 = max(0, cx-cw//2); y1 = max(0, cy-ch//2)
    x2 = min(W, cx+cw//2); y2 = min(H, cy+ch//2)
    mx = x.clone(); mx[:,:,y1:y2,x1:x2] = x[idx,:,y1:y2,x1:x2]
    return mx, y, y[idx], 1 - ((x2-x1)*(y2-y1))/(W*H)

def mc(crit, pred, ya, yb, l):
    return l * crit(pred, ya) + (1-l) * crit(pred, yb)

rsz = int(S['img'] * 1.15)
tr_tf = transforms.Compose([
    transforms.Resize((rsz, rsz)), transforms.RandomCrop(S['img']),
    transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.2),
    transforms.RandomRotation(20), transforms.ColorJitter(0.3, 0.3),
    transforms.ToTensor(), transforms.Normalize(S['mean'], S['std']),
    transforms.RandomErasing(0.2),
])
te_tf = transforms.Compose([
    transforms.Resize((S['img'], S['img'])),
    transforms.ToTensor(), transforms.Normalize(S['mean'], S['std']),
])

tr_ds = datasets.ImageFolder(TRAIN_DIR, transform=tr_tf)
te_ds = datasets.ImageFolder(TEST_DIR,  transform=te_tf)
tr_ld = DataLoader(tr_ds, batch_size=S['batch'], shuffle=True,  num_workers=2, pin_memory=True)
te_ld = DataLoader(te_ds, batch_size=S['batch'], shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(tr_ds)} | Test: {len(te_ds)}')

def build_model():
    m = CONFIG['model']
    if m == 'vit':
        mdl = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=NUM_CLASSES)
        for p in mdl.parameters(): p.requires_grad = False
        for i, blk in enumerate(mdl.blocks):
            if i >= len(mdl.blocks) - CONFIG['unfreeze']:
                for p in blk.parameters(): p.requires_grad = True
        for p in mdl.head.parameters(): p.requires_grad = True
        for p in mdl.norm.parameters(): p.requires_grad = True
    elif m == 'efficientnet_b2':
        mdl = models.efficientnet_b2(weights='IMAGENET1K_V1')
        for p in mdl.features.parameters(): p.requires_grad = False
        for nm, p in mdl.features.named_parameters():
            if any(f'{i}.' in nm for i in ['6','7','8']): p.requires_grad = True
        inf = mdl.classifier[1].in_features
        mdl.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(inf, 512), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(256, NUM_CLASSES)
        )
    else:
        mdl = models.efficientnet_b0(weights='IMAGENET1K_V1')
        for p in mdl.features.parameters(): p.requires_grad = False
        inf = mdl.classifier[1].in_features
        mdl.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(inf, 256), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(256, NUM_CLASSES)
        )
    return mdl.to(DEVICE)

model     = build_model()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)')

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=S['lr'], weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=S['lr']*0.01)

def train_ep():
    model.train(); tl = tc = tt = 0
    for imgs, lbs in tr_ld:
        imgs, lbs = imgs.to(DEVICE), lbs.to(DEVICE)
        r = np.random.rand()
        if CONFIG['mixup'] and r < 0.5:
            imgs, ya, yb, l = mixup(imgs, lbs)
            out = model(imgs); loss = mc(criterion, out, ya, yb, l)
        elif CONFIG['cutmix'] and r < 1.0:
            imgs, ya, yb, l = cutmix(imgs, lbs)
            out = model(imgs); loss = mc(criterion, out, ya, yb, l)
        else:
            out = model(imgs); loss = criterion(out, lbs)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        tl += loss.item()*imgs.size(0); tc += (out.argmax(1)==lbs).sum().item(); tt += lbs.size(0)
    return tl/tt, tc/tt

def val_ep():
    model.eval(); tl = tc = tt = 0; ap, al, apr = [], [], []
    with torch.no_grad():
        for imgs, lbs in te_ld:
            imgs, lbs = imgs.to(DEVICE), lbs.to(DEVICE)
            out = model(imgs); pr = torch.softmax(out, 1); loss = criterion(out, lbs)
            tl += loss.item()*imgs.size(0); tc += (out.argmax(1)==lbs).sum().item(); tt += lbs.size(0)
            ap.extend(out.argmax(1).cpu().numpy()); al.extend(lbs.cpu().numpy()); apr.extend(pr.cpu().numpy())
    return tl/tt, tc/tt, ap, al, apr

best_acc = 0; save_path = '/content/best_model.pth'
print(f'{"Epoch":<8}{"Train Acc":<12}{"Val Acc":<12}')
print('-' * 32)
for ep in range(1, CONFIG['epochs']+1):
    tl, ta = train_ep()
    vl, va, preds, labels, probs = val_ep()
    scheduler.step()
    st = ''
    if va > best_acc: best_acc = va; torch.save(model.state_dict(), save_path); st = 'Best!'
    print(f'{ep:<8}{ta:<12.4f}{va:<12.4f}{st}')
print(f'\nBest Accuracy: {best_acc*100:.2f}%')
if best_acc >= 0.90: print('90% TARGET ACHIEVED!')
else: print(f'Gap to 90%: {(0.90-best_acc)*100:.2f}%')

In [ ]:
#4 - Evaluate
model.load_state_dict(torch.load(save_path))
_, acc, preds, labels, probs = val_ep()
probs = np.array(probs); preds = np.array(preds); labels = np.array(labels)
auc = roc_auc_score(np.eye(NUM_CLASSES)[labels], probs, multi_class='ovr', average='macro')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{CONFIG["model"]} — {acc*100:.2f}% accuracy | AUC {auc:.4f}', fontsize=14, fontweight='bold')
cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
pcauc = [roc_auc_score((labels==i).astype(int), probs[:,i]) for i in range(NUM_CLASSES)]
axes[1].bar(CLASS_NAMES, pcauc, color=['#ef4444','#f97316','#22c55e','#6366f1'])
axes[1].set_ylim(0.8, 1.0); axes[1].set_title('Per-Class AUC-ROC')
for i, v in enumerate(pcauc): axes[1].text(i, v+0.002, f'{v:.3f}', ha='center')
plt.tight_layout(); plt.savefig('/content/evaluation.png', dpi=150); plt.show()
print(classification_report(labels, preds, target_names=CLASS_NAMES))
print(f'Overall AUC-ROC: {auc:.4f}')

cfg = {'model_name': CONFIG['model'], 'class_names': CLASS_NAMES, 'num_classes': NUM_CLASSES,
       'img_size': S['img'], 'best_val_acc': round(best_acc, 4), 'auc_roc': round(auc, 4)}
with open('/content/model_config.json', 'w') as f: json.dump(cfg, f, indent=2)
print('Saved model_config.json')

In [ ]:
#5 - Download
for fp, desc in [
    ('/content/best_model.pth',       'Model weights'),
    ('/content/model_config.json',    'Config'),
    ('/content/evaluation.png',       'Evaluation charts'),
]:
    if os.path.exists(fp): files.download(fp); print(f'Downloaded: {desc}')
print(f'Done! Best: {best_acc*100:.2f}%')